In [1]:
#Main Classifier
import json
import numpy as np
import pandas as pd
import random
from itertools import chain
from sentence_transformers import SentenceTransformer, util


random.seed(42)
np.random.seed(42)

# Load the CSV file
file_path = 'annotations.csv'  
df = pd.read_csv(file_path)
df = df.dropna()

data_as_list = df.values.tolist()

final_data = []
for data in data_as_list:
    if data[1] == 'W':
        gender = 'Female'
    else:
        gender = 'Male'
    final_data.append(["Post_text: "+data[2] + "\nResponse_text: "+data[3], data[4], gender])

formatted_data = np.array(final_data)

np.random.shuffle(formatted_data)

train_data = formatted_data[:6000]
ice_data = formatted_data[6000:12000]
test_data = formatted_data[12000:13200]



model = SentenceTransformer('bert-base-nli-mean-tokens')

def extract_questions(data):
    return [item[0] for item in data]

train_text = extract_questions(train_data)
in_context_text = extract_questions(ice_data)

train_embeddings = model.encode(train_text, convert_to_tensor=True)
in_context_embeddings = model.encode(in_context_text, convert_to_tensor=True)

similarity_matrix = util.cos_sim(train_embeddings, in_context_embeddings)

combinations = [
    ("2-sim-dissim", 2, {"similar": 1, "dissimilar": 1, "random": 0}),
    ("2-half-random", 2, {"similar": 1, "dissimilar": 0, "random": 1}),
    ("2-random", 2, {"similar": 0, "dissimilar": 0, "random": 2}),
    ("3-sim-dissim", 3, {"similar": 2, "dissimilar": 1, "random": 0}),
    ("3-half-random", 3, {"similar": 2, "dissimilar": 0, "random": 1}),
    ("3-random", 3, {"similar": 0, "dissimilar": 0, "random": 3}),
    ("4-sim-dissim", 4, {"similar": 2, "dissimilar": 2, "random": 0}),
    ("4-half-random", 4, {"similar": 2, "dissimilar": 0, "random": 2}),
    ("4-random", 4, {"similar": 0, "dissimilar": 0, "random": 4}),
    ("5-sim-dissim", 5, {"similar": 3, "dissimilar": 2, "random": 0}),
    ("5-half-random", 5, {"similar": 3, "dissimilar": 0, "random": 2}),
    ("5-random", 5, {"similar": 0, "dissimilar": 0, "random": 5}),
]

num_training = len(train_text)
groups = [list(range(num_training))[i:i + 500] for i in range(0, num_training, 500)]

final_indices = []
combination_types = []


for group, (comb_type, num_examples, counts) in zip(groups, combinations):
    for idx in group:
        similarities = similarity_matrix[idx]
        sorted_indices = similarities.argsort(descending=True)
        most_similar = sorted_indices[:counts["similar"]].tolist()
        most_dissimilar = sorted_indices[-counts["dissimilar"]:].tolist() if counts["dissimilar"] > 0 else []
        random_indices = random.sample(range(len(in_context_text)), counts["random"]) if counts["random"] > 0 else []
        selected_indices = list(chain(most_similar, most_dissimilar, random_indices))
        random.shuffle(selected_indices)
        final_indices.append(selected_indices)
        combination_types.append(comb_type)

merged_list_train = list(zip(final_indices, combination_types))



# Function to process test data for a dataset
def process_test_data(test_data, in_context_data, similarity_matrix, combinations):
    num_test_examples = len(test_data)
    group_size = num_test_examples // len(combinations)  # 22 examples per combination
    print(group_size)
    groups = [list(range(num_test_examples))[i:i + group_size] for i in range(0, num_test_examples, group_size)]

    final_indices = []
    combination_types = []

    for group, (comb_type, num_examples, counts) in zip(groups, combinations):
        for idx in group:
            similarities = similarity_matrix[idx]
            sorted_indices = similarities.argsort(descending=True)
            most_similar = sorted_indices[:counts["similar"]].tolist()
            most_dissimilar = sorted_indices[-counts["dissimilar"]:].tolist() if counts["dissimilar"] > 0 else []
            random_indices = random.sample(range(len(in_context_data)), counts["random"]) if counts["random"] > 0 else []
            selected_indices = list(chain(most_similar, most_dissimilar, random_indices))
            random.shuffle(selected_indices)
            final_indices.append(selected_indices)
            combination_types.append(comb_type)

    return final_indices, combination_types

test_text = extract_questions(test_data)
test_embeddings = model.encode(test_text, convert_to_tensor=True)
similarity_matrix_test = util.cos_sim(test_embeddings, in_context_embeddings)

final_indices_test, combination_types_test = process_test_data(
    test_text, in_context_text, similarity_matrix_test, combinations
)

# Merged list for test data in both datasets
merged_list_test = list(zip(final_indices_test, combination_types_test))



import numpy as np

import warnings
warnings.filterwarnings('ignore')



base_template = """Instruction:
You are an expert assistant trained to jointly predict the sentiment and the gender for the given input from social media post and its response. 

Possible types of sentiment are: 'Mixed', 'Negative', 'Neutral', and 'Positive'.  
Possible types of gender are: 'Male' and 'Female'.  

Output Format:
The output should be in the format: 'sentiment, gender'.

Examples:
"""

def put_example(data):
    
    template = """
{post}
Q: Predict the sentiment and the gender of the above post and response in the format sentiment, gender.
Answer: {sent}, {gend}
"""
    result_string = template.format(post=data[0], sent=data[1], gend=data[2])
    
    return result_string

def put_example_test(data):
    
    template = """
Now, solve for this example:
{post}
Q: Predict the sentiment and the gender of the above post and response in the format sentiment, gender.
Model Answer: {sent}, {gend}
"""
    result_string = template.format(post=data[0], sent=data[1], gend=data[2])
    
    return result_string

def put_example_test_for_test_data(data):
    template = """
Now, solve for this example:
{post}
Q: Predict the sentiment and the gender of the above post and response in the format sentiment, gender.
Model Answer: """
    result_string = template.format(post=data[0])
    
    return result_string





import ast
from datasets import Dataset, DatasetDict
def get_prompt(idx, data, Test=0):
    result = []
    temp = []
    target = []
    for index, id in enumerate(idx):
        temp_template = base_template
        indices_list = id[0]
        for i in range(len(indices_list)):
           example = put_example(ice_data[indices_list[i]])
           temp_template = temp_template + example
        
        if Test == 0:
            temp_template = temp_template + put_example_test(data[index])
        else:
            temp_template = temp_template + put_example_test_for_test_data(data[index])
        temp.append(temp_template)
        
        target.append((f'{data[index][1]}, {data[index][2]}'))
        result.append(f"{id[1]}")

    return {'input_text': temp, 'target_text': target, 'combination': result}
            
            
            
train_data = get_prompt(merged_list_train, train_data)
test_data = get_prompt(merged_list_test, test_data, Test=1)


train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)


#Convert DataFrames to Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Create DatasetDict
dataset_dict = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

100


In [2]:
# Separate the data into male and female
male_entries = [entry for entry in ice_data if entry[2] == 'Male']
female_entries = [entry for entry in ice_data if entry[2] == 'Female']
print(len(male_entries))
print(len(female_entries))

3002
2998


In [3]:
# Unbiased Classifier

import json
import numpy as np
import pandas as pd
import random
from itertools import chain
from sentence_transformers import SentenceTransformer, util


random.seed(42)
np.random.seed(42)

# Load the CSV file
file_path = 'annotations.csv'  
df = pd.read_csv(file_path)
df = df.dropna()

data_as_list = df.values.tolist()

final_data = []
for data in data_as_list:
    if data[1] == 'W':
        gender = 'Female'
    else:
        gender = 'Male'
    final_data.append(["Post_text: "+data[2] + "\nResponse_text: "+data[3], data[4], gender])

formatted_data = np.array(final_data)

np.random.shuffle(formatted_data)

train_data = formatted_data[:6000]
ice_data = formatted_data[6000:12000]
test_data = formatted_data[12000:13200]



model = SentenceTransformer('bert-base-nli-mean-tokens')

def extract_questions(data):
    return [item[0] for item in data]

train_text = extract_questions(train_data)
in_context_text = extract_questions(ice_data)

train_embeddings = model.encode(train_text, convert_to_tensor=True)
in_context_embeddings = model.encode(in_context_text, convert_to_tensor=True)

similarity_matrix = util.cos_sim(train_embeddings, in_context_embeddings)

combinations = [
    ("2-sim-dissim", 2, {"similar": 1, "dissimilar": 1, "random": 0}),
    ("2-half-random", 2, {"similar": 1, "dissimilar": 0, "random": 1}),
    ("2-random", 2, {"similar": 0, "dissimilar": 0, "random": 2}),
    ("3-sim-dissim", 3, {"similar": 2, "dissimilar": 1, "random": 0}),
    ("3-half-random", 3, {"similar": 2, "dissimilar": 0, "random": 1}),
    ("3-random", 3, {"similar": 0, "dissimilar": 0, "random": 3}),
    ("4-sim-dissim", 4, {"similar": 2, "dissimilar": 2, "random": 0}),
    ("4-half-random", 4, {"similar": 2, "dissimilar": 0, "random": 2}),
    ("4-random", 4, {"similar": 0, "dissimilar": 0, "random": 4}),
    ("5-sim-dissim", 5, {"similar": 3, "dissimilar": 2, "random": 0}),
    ("5-half-random", 5, {"similar": 3, "dissimilar": 0, "random": 2}),
    ("5-random", 5, {"similar": 0, "dissimilar": 0, "random": 5}),
]

num_training = len(train_text)
groups = [list(range(num_training))[i:i + 500] for i in range(0, num_training, 500)]

final_indices = []
combination_types = []


for group, (comb_type, num_examples, counts) in zip(groups, combinations):
    for idx in group:
        similarities = similarity_matrix[idx]
        sorted_indices = similarities.argsort(descending=True)
        most_similar = sorted_indices[:counts["similar"]].tolist()
        most_dissimilar = sorted_indices[-counts["dissimilar"]:].tolist() if counts["dissimilar"] > 0 else []
        random_indices = random.sample(range(len(in_context_text)), counts["random"]) if counts["random"] > 0 else []
        selected_indices = list(chain(most_similar, most_dissimilar, random_indices))
        random.shuffle(selected_indices)
        final_indices.append(selected_indices)
        combination_types.append(comb_type)

merged_list_train = list(zip(final_indices, combination_types))



# Function to process test data for a dataset
def process_test_data(test_data, in_context_data, similarity_matrix, combinations):
    num_test_examples = len(test_data)
    group_size = num_test_examples // len(combinations)  # 22 examples per combination
    print(group_size)
    groups = [list(range(num_test_examples))[i:i + group_size] for i in range(0, num_test_examples, group_size)]

    final_indices = []
    combination_types = []

    for group, (comb_type, num_examples, counts) in zip(groups, combinations):
        for idx in group:
            similarities = similarity_matrix[idx]
            sorted_indices = similarities.argsort(descending=True)
            most_similar = sorted_indices[:counts["similar"]].tolist()
            most_dissimilar = sorted_indices[-counts["dissimilar"]:].tolist() if counts["dissimilar"] > 0 else []
            random_indices = random.sample(range(len(in_context_data)), counts["random"]) if counts["random"] > 0 else []
            selected_indices = list(chain(most_similar, most_dissimilar, random_indices))
            random.shuffle(selected_indices)
            final_indices.append(selected_indices)
            combination_types.append(comb_type)

    return final_indices, combination_types

test_text = extract_questions(test_data)
test_embeddings = model.encode(test_text, convert_to_tensor=True)
similarity_matrix_test = util.cos_sim(test_embeddings, in_context_embeddings)

final_indices_test, combination_types_test = process_test_data(
    test_text, in_context_text, similarity_matrix_test, combinations
)

# Merged list for test data in both datasets
merged_list_test = list(zip(final_indices_test, combination_types_test))



import numpy as np

import warnings
warnings.filterwarnings('ignore')



base_template = """Instruction:
You are an expert assistant trained to jointly predict the sentiment and the gender for the given input from social media post and its response. 

Possible types of sentiment are: 'Mixed', 'Negative', 'Neutral', and 'Positive'.  
Possible types of gender are: 'Male' and 'Female'.  

Output Format:
The output should be in the format: 'sentiment, gender'.

Examples:
"""

def put_example(data):
    
    template = """
{post}
Q: Predict the sentiment and the gender of the above post and response in the format sentiment, gender.
Answer: {sent}, {gend}
"""
    result_string = template.format(post=data[0], sent=data[1], gend=data[2])
    
    return result_string

def put_example_test(data):
    
    template = """
Now, solve for this example:
{post}
Q: Predict the sentiment and the gender of the above post and response in the format sentiment, gender.
Model Answer: {sent}, {gend}
"""
    result_string = template.format(post=data[0], sent=data[1], gend=data[2])
    
    return result_string

def put_example_test_for_test_data(data):
    template = """
Now, solve for this example:
{post}
Q: Predict the sentiment and the gender of the above post and response in the format sentiment, gender.
Model Answer: """
    result_string = template.format(post=data[0])
    
    return result_string





import ast
from datasets import Dataset, DatasetDict
def get_prompt(idx, data, Test=0):
    result = []
    temp = []
    target = []
    for index, id in enumerate(idx):
        temp_template = base_template
        indices_list = id[0]
        for i in range(len(indices_list)):
           random_entry_male = random.choice(female_entries)
           example = put_example(random_entry_male)
           temp_template = temp_template + example
        
        if Test == 0:
            temp_template = temp_template + put_example_test(data[index])
        else:
            temp_template = temp_template + put_example_test_for_test_data(data[index])
        temp.append(temp_template)
        
        target.append((f'{data[index][1]}, {data[index][2]}'))
        result.append(f"{id[1]}")

    return {'input_text': temp, 'target_text': target, 'combination': result}
            
            
            
train_data = get_prompt(merged_list_train, train_data)
test_data = get_prompt(merged_list_test, test_data, Test=1)


train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)


#Convert DataFrames to Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Create DatasetDict
dataset_dict = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

100


In [11]:
test_df

,input_text,target_text,combination
0,Instruction:\nYou are an expert assistant trai...,"Neutral, Male",5-random
1,Instruction:\nYou are an expert assistant trai...,"Neutral, Male",4-random
2,Instruction:\nYou are an expert assistant trai...,"Positive, Female",2-half-random
3,Instruction:\nYou are an expert assistant trai...,"Positive, Female",3-half-random
4,Instruction:\nYou are an expert assistant trai...,"Neutral, Male",2-sim-dissim
...,...,...,...
1195,Instruction:\nYou are an expert assistant trai...,"Positive, Female",5-half-random
1196,Instruction:\nYou are an expert assistant trai...,"Mixed, Male",5-half-random
1197,Instruction:\nYou are an expert assistant trai...,"Positive, Male",5-random
1198,Instruction:\nYou are an expert assistant trai...,"Positive, Male",4-random


In [16]:
import numpy as np
import pandas as pd

# Sample NumPy array (replace this with your actual data)
# Assuming your data is structured as [bio, profession, gender, raw_bio]
data = test_df.values

data = data[:, 1]
data = [item.split(', ') for item in data]
data = np.array(data)

# Extract sentiment and gender columns
sentiment = data[:, 0]  # Column index for Sentiment
genders = data[:, 1]      # Column index for gender

# Get unique sentiment and genders
unique_sentiment = np.unique(sentiment)
unique_genders = np.unique(genders)

# Create a mapping from sentiment/gender to index
sentiment_to_index = {prof: idx for idx, prof in enumerate(unique_sentiment)}
gender_to_index = {gend: idx for idx, gend in enumerate(unique_genders)}

# Initialize the count matrix with zeros
count_matrix = np.zeros((len(unique_sentiment), len(unique_genders)), dtype=int)

# Fill the count matrix with counts
for prof, gend in zip(sentiment, genders):
    prof_idx = sentiment_to_index[prof]
    gend_idx = gender_to_index[gend]
    count_matrix[prof_idx, gend_idx] += 1

# Create a DataFrame from the count matrix
gender_sentiment_count = pd.DataFrame(count_matrix, index=unique_sentiment, columns=unique_genders)

# Print the DataFrame
print("Count DataFrame (rows: sentiment, columns: genders):")
print(gender_sentiment_count)

# Print the total number of unique sentiment
print(f"Total number of unique sentiment: {len(unique_sentiment)}")

# Identify minority combinations
minority_indices = []
for sent in gender_sentiment_count.index:
    male_count = gender_sentiment_count.loc[sent, 'Male']
    female_count = gender_sentiment_count.loc[sent, 'Female']
    if male_count <= female_count:
        minority_indices.append((sent, 'Male'))
    if female_count <= male_count:
        minority_indices.append((sent, 'Female'))

Count DataFrame (rows: sentiment, columns: genders):
          Female  Male
Mixed         43    69
Negative      84   116
Neutral      135   165
Positive     297   291
Total number of unique sentiment: 4


Val_data

Count DataFrame (rows: professions, columns: genders):
                    F   M
accountant         34  66
architect          26  74
attorney           38  62
chiropractor       31  69
comedian           23  77
composer           14  86
dentist            28  72
dietitian          89  11
dj                  7  93
filmmaker          34  66
interior_designer  80  20
journalist         53  47
model              80  20
nurse              92   8
painter            46  54
paralegal          80  20
pastor             28  72
personal_trainer   44  56
photographer       37  63
physician          52  48
poet               54  46
professor          46  54
psychologist       66  34
rapper              5  95
software_engineer  14  86
surgeon            13  87
teacher            65  35
yoga_teacher       79  21
Total number of unique professions: 28


In [17]:
minority_indices

[('Mixed', 'Female'),
 ('Negative', 'Female'),
 ('Neutral', 'Female'),
 ('Positive', 'Male')]

In [18]:
from peft import LoraConfig, PeftConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub.hf_api import HfFolder
import torch
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)



HfFolder.save_token(os.environ.get("HF_TOKEN"))
model_id = "meta-llama/Llama-2-7b-chat-hf"

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config,  trust_remote_code=True,)

model.config.use_cache = True # silence the warnings
model.config.pretraining_tp = 1
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
#model.resize_token_embeddings(len(tokenizer))

model = PeftModel.from_pretrained(model, 'fine_tuned_llama2_rtgen_10epoch')


tokenizer.padding_side = "left"
tokenizer.pad_token_id = tokenizer.bos_token_id
tokenizer.pad_token = tokenizer.bos_token
model.config.pad_token_id = tokenizer.bos_token_id
model = model.bfloat16()
#model.resize_token_embeddings(len(tokenizer))


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
`low_cpu_mem_usage` was None, now set to True since model is quantized.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment var

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
from peft import PeftModel

# Load the fine-tuned model
#model = PeftModel.from_pretrained(model, 'fine_tuned_llama2_test')

# Access the PEFT config to check rank and alpha values
peft_config = model.peft_config

for layer_name, config in peft_config.items():
    print(f"Layer: {layer_name}")
    print(f"LoRA Rank: {config.r}")
    print(f"LoRA Alpha: {config.lora_alpha}")

Layer: default
LoRA Rank: 8
LoRA Alpha: 64


In [7]:
len(tokenizer)

32000

In [25]:
import torch
from tqdm import tqdm
import pickle
import os

# Ensure you're in evaluation mode
model.eval()

# Define the batch size
batch_size = 4  # Adjust based on your GPU memory

def infer_batch_with_checkpoint(model, tokenizer, input_texts, batch_size, max_input_length, checkpoint_file='checkpoint_custom_loss_main_classifier_no_flip.pkl'):
    predictions = []
    start_idx = 0

    # Load checkpoint if it exists
    # if os.path.exists(checkpoint_file):
    #     with open(checkpoint_file, 'rb') as f:
    #         checkpoint = pickle.load(f)
    #         start_idx = checkpoint['last_idx']
    #         predictions = checkpoint['predictions']
    #         print(f"Resuming from batch index {start_idx}")

    num_batches = (len(input_texts) + batch_size - 1) // batch_size  # Calculate number of batches

    with torch.no_grad():
        for i in tqdm(range(start_idx // batch_size, num_batches), desc="Processing Batches", unit="batch"):
            start_idx = i * batch_size
            end_idx = min(start_idx + batch_size, len(input_texts))
            batch = input_texts[start_idx:end_idx]
            
            # Tokenize the batch
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_input_length)
            input_ids = inputs["input_ids"].to(model.device)
            attention_mask = inputs["attention_mask"].to(model.device)
            
            # Forward pass
            outputs = model.generate(input_ids, attention_mask=attention_mask, max_new_tokens=10, num_return_sequences=1)
            
            # Decode the predictions
            batch_predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            for pred in batch_predictions:
                
                if 'Model Answer:' in pred:
                    _, result = pred.split('Model Answer: ', 1)  # Split and get part after the delimiter
                    predictions.append(result.strip())
                else:
                    predictions.append(pred.strip())
            
            # Save checkpoint
            checkpoint = {
                'last_idx': end_idx,
                'predictions': predictions
            }
            with open(checkpoint_file, 'wb') as f:
                pickle.dump(checkpoint, f)
    
    return predictions

# Test dataset
test_data = test_dataset['input_text']

# Get predictions with checkpointing
predictions = infer_batch_with_checkpoint(model, tokenizer, test_data, batch_size, max_input_length=1024)

# Optionally, compare with the true labels if needed
test_labels = test_dataset['target_text']

Processing Batches:   0%|          | 0/300 [00:00<?, ?batch/s]

Processing Batches: 100%|██████████| 300/300 [08:21<00:00,  1.67s/batch]


In [23]:
predictions

['Neutral, Male',
 'Neutral, Female',
 'Positive, Female',
 'Positive, Male',
 'Positive, Male']

In [26]:
cleaned_res = [item.split('\n')[0] for item in predictions]
split_data_generated = [item.split(', ') for item in cleaned_res]
split_data_true = [item.split(', ') for item in test_labels]


In [27]:
gc = 0


gen_gend = []
true_gend = []

prof_to_match = [
    'psychologist', 'poet', 'photographer', 'nurse', 'software_engineer',
    'comedian', 'pastor', 'architect', 'chiropractor', 'dentist', 'model',
    'interior_designer', 'teacher', 'accountant', 'rapper', 'yoga_teacher',
    'paralegal', 'surgeon', 'painter', 'composer', 'dj', 'personal_trainer',
    'physician', 'journalist', 'dietitian', 'filmmaker', 'attorney', 'professor'
]
gend_to_match = ['Male', 'Female']
for i, txt in enumerate(predictions):
    flag = 0
    for gend in gend_to_match:
        if gend in txt:
            
            if(gend.lower()==split_data_true[i][1].lower() or (gend.lower()=='fem' and split_data_true[i][1] == 'Female') or (gend.lower()=='feale' and split_data_true[i][1] == 'Female')):
                gc+=1
                gen_gend.append(split_data_true[i][1])
                true_gend.append(split_data_true[i][1])
                flag=1
                break
    if flag == 0: 
        gen_gend.append('Male' if split_data_true[i][1] == 'Female' else 'Female')
        true_gend.append(split_data_true[i][1])
print(gc/len(predictions))
    

0.7266666666666667


In [12]:
print(len(true_gend))
print(len(gen_gend))


2800
2800


In [8]:
from sentence_transformers import SentenceTransformer, util
bert_model = SentenceTransformer('all-MiniLM-L6-v2')

professions_to_match = [
    'psychologist', 'poet', 'photographer', 'nurse', 'software_engineer',
    'comedian', 'pastor', 'architect', 'chiropractor', 'dentist', 'model',
    'interior_designer', 'teacher', 'accountant', 'rapper', 'yoga_teacher',
    'paralegal', 'surgeon', 'painter', 'composer', 'dj', 'personal_trainer',
    'physician', 'journalist', 'dietitian', 'filmmaker', 'attorney', 'professor'
]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [9]:
len(split_data_generated)

2800

In [10]:
profession_correct = 0
#gender_correct = 0
c = 0
true_prof = []
gen_prof = []

i = 0

for gen, true in zip(split_data_generated, split_data_true):
    #print(i)
    i+=1
    flag = 0
    true_prof.append(true[0])
    #true_gend.append(true[1])
    temp = 'NA'
    if len(gen) < 2: 
        if (isinstance(gen, list)): gen = gen[0]
        gen_profession = gen
        true_profession, _ = true
        #print(gen, true)
        c+=1
        # gen_prof.append('NA')
        # if true[1] == 'Male':
        #     gen_gend.append('Female')
        # else:
        #     gen_gend.append('Male')
    elif(len(gen) > 2):
        #print(gen)
        gen = gen[-2:]
        
        gen_profession, _ = gen
        true_profession, _ = true
    else: 
        gen_profession, _ = gen
        true_profession, _ = true
    
    #print(true_gender, gen_gender)
    # Profession check
    if true_profession.lower() in gen_profession.lower():
        gen_prof.append(true_profession)
        profession_correct += 1
        flag = 1
    elif(len(gen_profession)!= 0):
        # print(true_profession,'|', gen_profession)
        
        if gen_profession not in professions_to_match:
            similarity_array=[]
            for profession in professions_to_match:
                word1=gen_profession
                word2=profession
                embeddings1 = bert_model.encode(word1, convert_to_tensor=True)
                embeddings2 = bert_model.encode(word2, convert_to_tensor=True)
                similarity_array.append((util.cos_sim(embeddings1, embeddings2).item(),profession))
            sorted_similarity_array = sorted(similarity_array, key=lambda x: x[0], reverse=True)
            temp = sorted_similarity_array[0][1]
            #print(sorted_similarity_array[0][1])
            if sorted_similarity_array[0][1] == true_profession.lower():
                #print(sorted_similarity_array[0][1], gen_profession)
                gen_prof.append(true_profession)
                profession_correct += 1
                flag = 1
        else:
            temp = gen_profession

    if flag == 0:
        #print(true_profession, gen_profession)
        gen_prof.append(temp)
        #print(true_profession,'|', gen_profession, '|', temp)

    # Gender check
    # if ': Male' in gen_gender or ': Fem' in gen_gender:
    #     gen_gender = gen_gender.split(': ')[1]
    #     #print(gen_gender)
    # if gen_gender == true_gender or (gen_gender == 'Fem' and true_gender == 'Female'):
    #     gender_correct += 1
    #     #print(gen_gender, true_gender)
    #     gen_gend.append(true_gender)
        
    # else:
    #     #print(gen_gender, true_gender)
    #     if true[1] == 'Male':
    #         gen_gend.append('Female')
    #     else:
    #         gen_gend.append('Male')


# Total number of samples
total_samples = len(split_data_true)

# Calculate accuracy
profession_accuracy = profession_correct / total_samples
#gender_accuracy = gender_correct / total_samples

print(f"Profession Accuracy: {profession_accuracy:.5f}")
#print(f"Gender Accuracy: {gender_accuracy:.5f}")

Profession Accuracy: 0.96000


In [11]:
print(len(gen_prof))
print(len(true_prof))

2800
2800


In [34]:
df = pd.read_csv('./predictions/fine_tuned_llama2_10epochs.csv')

# Convert the columns back into lists
true_prof = df['True_Profession'].tolist()
gen_prof = df['Generated_Profession'].tolist()
true_gend = df['True_Gender'].tolist()
gen_gend = df['Generated_Gender'].tolist()


In [35]:
from collections import defaultdict

# Create dictionaries to count correct and total responses
correct_counts = defaultdict(int)
total_counts = defaultdict(int)

# Combine the lists into tuples (true_profession, true_gender, predicted_profession)
for tg, tp, pp in zip(true_gend, true_prof, gen_prof):
    key = (tp, tg)  # Unique combination of (true_profession, true_gender)
    total_counts[key] += 1
    if tp == pp:  # Check if the profession matches
        correct_counts[key] += 1

# Calculate accuracy for each unique (true_profession, true_gender) combination
accuracy = {key: correct_counts[key] / total_counts[key] if total_counts[key] > 0 else 0 for key in total_counts}

# Display results
print("Accuracy for each unique (true_profession, true_gender) combination:")
for key in accuracy:
    print(f"Combination {key}: Accuracy = {accuracy[key]:.2f}")


Accuracy for each unique (true_profession, true_gender) combination:
Combination ('Neutral', 'Male'): Accuracy = 0.66
Combination ('Positive', 'Female'): Accuracy = 0.92
Combination ('Positive', 'Male'): Accuracy = 0.86
Combination ('Negative', 'Female'): Accuracy = 0.69
Combination ('Neutral', 'Female'): Accuracy = 0.49
Combination ('Negative', 'Male'): Accuracy = 0.65
Combination ('Mixed', 'Male'): Accuracy = 0.06
Combination ('Mixed', 'Female'): Accuracy = 0.14


In [36]:
# Prepare data for CSV
data = []
for key in accuracy:
    true_profession, true_gender = key
    acc = accuracy[key]
    correct = correct_counts[key]
    total = total_counts[key]
    data.append([true_profession, true_gender, acc, correct, total])

# Create DataFrame
df = pd.DataFrame(data, columns=['True Sentiment', 'True Gender', 'Accuracy', 'Correct Count', 'Total Count'])

# Save DataFrame to CSV
df.to_csv('./predictions/accuracy_summary_fine_tuned_llama2_10epochs.csv', index=False)



In [38]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load the two CSV files into DataFrames
accuracy_summary_unbiased_classifier_male = pd.read_csv('./predictions/accuracy_summary_unbiased_classifier_male.csv')
accuracy_summary_unbiased_classifier_female = pd.read_csv('./predictions/accuracy_summary_unbiased_classifier_female.csv')
accuracy_summary = pd.read_csv('./predictions/accuracy_summary_fine_tuned_llama2_10epochs.csv')

# Print the lengths of the DataFrames
print(f"Length of DataFrame df1: {len(accuracy_summary_unbiased_classifier_male)}")
print(f"Length of DataFrame df2: {len(accuracy_summary)}")
print(accuracy_summary.head())


# Pivot the DataFrame to get a matrix format suitable for a heatmap
# heatmap_data = accuracy_summary.pivot_table(
#     index='True Profession',
#     columns='True Gender',
#     values='Accuracy',
#     aggfunc='mean'  # Aggregates values if there are multiple entries
# )

# # Plot the heatmap
# plt.figure(figsize=(10, 6))
# sns.heatmap(heatmap_data, annot=True, cmap='YlGnBu', fmt='.2f')
# plt.title('Accuracy Heatmap by Profession and Gender')
# plt.xlabel('True Gender')
# plt.ylabel('True Profession')
# plt.show()

# Merge DataFrames on 'True Profession' and 'True Gender'
merged_df_male = pd.merge(accuracy_summary, accuracy_summary_unbiased_classifier_male, on=['True Sentiment', 'True Gender'], suffixes=('', '_unbiased_classifier'))

merged_df_female = pd.merge(accuracy_summary, accuracy_summary_unbiased_classifier_female, on=['True Sentiment', 'True Gender'], suffixes=('', '_unbiased_classifier'))


Length of DataFrame df1: 8
Length of DataFrame df2: 8
  True Sentiment True Gender  Accuracy  Correct Count  Total Count
0        Neutral        Male  0.660606            109          165
1       Positive      Female  0.922559            274          297
2       Positive        Male  0.855670            249          291
3       Negative      Female  0.690476             58           84
4        Neutral      Female  0.488889             66          135


In [40]:
results_male = []
results_female = []


for idx, row in merged_df_female.iterrows():
    # Get the current combination's counts
    current_prof = row['True Sentiment']
    current_gend = row['True Gender']
    current_correct_count = row['Correct Count']
    current_total_count = row['Total Count']
    current_correct_count_unbiased = row['Correct Count_unbiased_classifier'] 
    current_total_count_unbiased = row['Total Count_unbiased_classifier']
    
     # Compute counts for all other combinations
    other_correct_count = merged_df_female[(merged_df_female['True Sentiment'] != current_prof) & (merged_df_female['True Gender'] != current_gend)]['Correct Count'].sum()
    other_total_count = merged_df_female[(merged_df_female['True Sentiment'] != current_prof) & (merged_df_female['True Gender'] != current_gend)]['Total Count'].sum()
    
    other_correct_count_unbiased = merged_df_female[(merged_df_female['True Sentiment'] != current_prof) & (merged_df_female['True Gender'] != current_gend)]['Correct Count_unbiased_classifier'].sum()
    other_total_count_unbiased = merged_df_male[(merged_df_female['True Sentiment'] != current_prof) & (merged_df_female['True Gender'] != current_gend)]['Total Count_unbiased_classifier'].sum()
    
    # Calculate the metric
    accuracy_minor_group = (current_correct_count + other_correct_count) / (current_total_count + other_total_count) if (current_total_count + other_total_count) > 0 else 0
    accuracy_minor_group_unbiased = (current_correct_count_unbiased + other_correct_count_unbiased) / (current_total_count_unbiased + other_total_count_unbiased) if (current_total_count_unbiased + other_total_count_unbiased) > 0 else 0
    
    spuriousness_score_minor_group = abs(1 - (accuracy_minor_group/accuracy_minor_group_unbiased))
    
    # Store the results
    results_female.append({
        'True Sentiment': current_prof,
        'True Gender': current_gend,
        'Spuriousness Score': spuriousness_score_minor_group,
    })

for idx, row in merged_df_male.iterrows():
    # Get the current combination's counts
    current_prof = row['True Sentiment']
    current_gend = row['True Gender']
    current_correct_count = row['Correct Count']
    current_total_count = row['Total Count']
    current_correct_count_unbiased = row['Correct Count_unbiased_classifier'] 
    current_total_count_unbiased = row['Total Count_unbiased_classifier']
    
     # Compute counts for all other combinations
    other_correct_count = merged_df_male[(merged_df_male['True Sentiment'] != current_prof) & (merged_df_male['True Gender'] != current_gend)]['Correct Count'].sum()
    other_total_count = merged_df_male[(merged_df_male['True Sentiment'] != current_prof) & (merged_df_male['True Gender'] != current_gend)]['Total Count'].sum()
    
    other_correct_count_unbiased = merged_df_male[(merged_df_male['True Sentiment'] != current_prof) & (merged_df_male['True Gender'] != current_gend)]['Correct Count_unbiased_classifier'].sum()
    other_total_count_unbiased = merged_df_male[(merged_df_male['True Sentiment'] != current_prof) & (merged_df_male['True Gender'] != current_gend)]['Total Count_unbiased_classifier'].sum()
    
    # Calculate the metric
    accuracy_minor_group = (current_correct_count + other_correct_count) / (current_total_count + other_total_count) if (current_total_count + other_total_count) > 0 else 0
    accuracy_minor_group_unbiased = (current_correct_count_unbiased + other_correct_count_unbiased) / (current_total_count_unbiased + other_total_count_unbiased) if (current_total_count_unbiased + other_total_count_unbiased) > 0 else 0
    
    spuriousness_score_minor_group = abs(1 - (accuracy_minor_group/accuracy_minor_group_unbiased))
    
    # Store the results
    results_male.append({
        'True Sentiment': current_prof,
        'True Gender': current_gend,
        'Spuriousness Score': spuriousness_score_minor_group,
    })
    
    
# Convert results to DataFrame
results_df_male = pd.DataFrame(results_male)
results_df_female = pd.DataFrame(results_female)

filter_df = pd.DataFrame(minority_indices, columns=['True Sentiment', 'True Gender'])

# Perform filtering based on both profession and gender
filtered_df_male = results_df_male.merge(filter_df, on=['True Sentiment', 'True Gender'], how='inner')
filtered_df_female = results_df_female.merge(filter_df, on=['True Sentiment', 'True Gender'], how='inner')

combined_df = pd.concat([filtered_df_male, filtered_df_female])

# Group by 'True Profession' and 'True Gender', and get the maximum 'Spuriousness Score'
max_spuriousness_df = combined_df.groupby(['True Sentiment', 'True Gender'], as_index=False)['Spuriousness Score'].max()

mean_spuriousness = max_spuriousness_df['Spuriousness Score'].max()


# Print the mean
print(f'Max Mean Spuriousness Score: {mean_spuriousness}')

min_spuriousness_df = combined_df.groupby(['True Sentiment', 'True Gender'], as_index=False)['Spuriousness Score'].min()

mean_spuriousness = min_spuriousness_df['Spuriousness Score'].max()


# Print the mean
print(f'Min Mean Spuriousness Score: {mean_spuriousness}')

Max Mean Spuriousness Score: 0.0370370370370372
Min Mean Spuriousness Score: 0.030516431924882736
